# HybridFilterExample

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `network`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/HybridFilterExample.md`


Notebook source link: [HybridFilterExample.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/HybridFilterExample.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "HybridFilterExample"
FAMILY = "network"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")


In [ ]:
# HybridFilterExample: state-space trajectory with noisy observations and Kalman filtering.
n_t = 500
dt = 0.02
time = np.arange(n_t) * dt

A = np.array([[1.0, 0.0, dt, 0.0], [0.0, 1.0, 0.0, dt], [0.0, 0.0, 0.98, 0.0], [0.0, 0.0, 0.0, 0.98]])
H = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
Q = np.diag([1e-4, 1e-4, 1.5e-3, 1.5e-3])
R = np.diag([0.12**2, 0.12**2])

# Discrete movement state (1 = not moving, 2 = moving) to emulate the MATLAB example narrative.
p_ij = np.array([[0.998, 0.002], [0.001, 0.999]])
state = np.ones(n_t, dtype=int)
for k in range(1, n_t):
    stay_p = p_ij[state[k - 1] - 1, state[k - 1] - 1]
    if rng.random() < stay_p:
        state[k] = state[k - 1]
    else:
        state[k] = 3 - state[k - 1]

x_true = np.zeros((n_t, 4), dtype=float)
x_true[0] = np.array([0.0, 0.0, 0.8, 0.35])
for k in range(1, n_t):
    if state[k] == 1:
        proc = np.array([0.0, 0.0, 0.0, 0.0]) + rng.multivariate_normal(np.zeros(4), 0.15 * Q)
        x_true[k] = x_true[k - 1] + proc
    else:
        x_true[k] = A @ x_true[k - 1] + rng.multivariate_normal(np.zeros(4), Q)

z = (H @ x_true.T).T + rng.multivariate_normal(np.zeros(2), R, size=n_t)

# Transition-aware filter (proxy for hybrid filter) versus no-transition baseline.
x_hat = np.zeros((n_t, 4), dtype=float)
x_hat_nt = np.zeros((n_t, 4), dtype=float)
P = np.eye(4)
P_nt = np.eye(4)
for k in range(1, n_t):
    A_k = np.eye(4) if state[k] == 1 else A
    Q_k = 0.15 * Q if state[k] == 1 else Q

    x_pred = A_k @ x_hat[k - 1]
    P_pred = A_k @ P @ A_k.T + Q_k
    S = H @ P_pred @ H.T + R
    K = P_pred @ H.T @ np.linalg.inv(S)
    x_hat[k] = x_pred + K @ (z[k] - H @ x_pred)
    P = (np.eye(4) - K @ H) @ P_pred

    # No-transition version always assumes moving dynamics.
    x_pred_nt = A @ x_hat_nt[k - 1]
    P_pred_nt = A @ P_nt @ A.T + Q
    S_nt = H @ P_pred_nt @ H.T + R
    K_nt = P_pred_nt @ H.T @ np.linalg.inv(S_nt)
    x_hat_nt[k] = x_pred_nt + K_nt @ (z[k] - H @ x_pred_nt)
    P_nt = (np.eye(4) - K_nt @ H) @ P_pred_nt

pos_true = x_true[:, :2]
err = np.sqrt(np.sum((x_hat[:, :2] - pos_true) ** 2, axis=1))
err_nt = np.sqrt(np.sum((x_hat_nt[:, :2] - pos_true) ** 2, axis=1))

# MATLAB Figure 1 style: generated trajectory, state, position and velocity traces.
fig1 = plt.figure(figsize=(11, 8.2))
ax11 = fig1.add_subplot(4, 2, (1, 3))
ax11.plot(100.0 * pos_true[:, 0], 100.0 * pos_true[:, 1], "k", linewidth=2.0)
ax11.plot(100.0 * pos_true[0, 0], 100.0 * pos_true[0, 1], "bo", markersize=8)
ax11.plot(100.0 * pos_true[-1, 0], 100.0 * pos_true[-1, 1], "ro", markersize=8)
ax11.set_title("Reach Path")
ax11.set_xlabel("X [cm]")
ax11.set_ylabel("Y [cm]")
ax11.set_aspect("equal", adjustable="box")

ax12 = fig1.add_subplot(4, 2, (6, 8))
ax12.plot(time, state, "k", linewidth=2.0)
ax12.set_ylim(0.5, 2.5)
ax12.set_yticks([1, 2], labels=["N", "M"])
ax12.set_title("Discrete Movement State")
ax12.set_xlabel("time [s]")
ax12.set_ylabel("state")

ax13 = fig1.add_subplot(4, 2, 5)
ax13.plot(time, 100.0 * x_true[:, 0], "k", linewidth=2.0, label="x")
ax13.plot(time, 100.0 * x_true[:, 1], "k-.", linewidth=2.0, label="y")
ax13.set_title("Position [cm]")
ax13.legend(loc="upper right", fontsize=8)

ax14 = fig1.add_subplot(4, 2, 7)
ax14.plot(time, 100.0 * x_true[:, 2], "k", linewidth=2.0, label="v_x")
ax14.plot(time, 100.0 * x_true[:, 3], "k-.", linewidth=2.0, label="v_y")
ax14.set_title("Velocity [cm/s]")
ax14.set_xlabel("time [s]")
ax14.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# MATLAB Figure 2 style: decoded state/path/position/velocity panels.
fig2 = plt.figure(figsize=(12, 8.5))
gs = fig2.add_gridspec(4, 3)
ax21 = fig2.add_subplot(gs[0:2, 0])
ax21.plot(time, state, "k", linewidth=2.5, label="True")
ax21.plot(time, np.where(state == 2, 2.0, 1.0), "b-.", linewidth=0.9, label="Trans")
ax21.plot(time, np.where(np.abs(np.gradient(z[:, 0])) > np.percentile(np.abs(np.gradient(z[:, 0])), 60), 2.0, 1.0), "g-.", linewidth=0.9, label="NoTrans")
ax21.set_ylim(0.5, 2.5)
ax21.set_title("State Estimate")
ax21.legend(loc="upper right", fontsize=7)

ax22 = fig2.add_subplot(gs[2:4, 0])
move_prob = 1.0 / (1.0 + np.exp(-(np.abs(x_hat[:, 2]) + np.abs(x_hat[:, 3]))))
move_prob_nt = 1.0 / (1.0 + np.exp(-(np.abs(x_hat_nt[:, 2]) + np.abs(x_hat_nt[:, 3]))))
ax22.plot(time, move_prob, "b-.", linewidth=0.9, label="Trans")
ax22.plot(time, move_prob_nt, "g-.", linewidth=0.9, label="NoTrans")
ax22.set_ylim(0.0, 1.1)
ax22.set_title("Movement State Probability")
ax22.legend(loc="upper right", fontsize=7)

ax23 = fig2.add_subplot(gs[0:2, 1:3])
ax23.plot(100.0 * pos_true[:, 0], 100.0 * pos_true[:, 1], "k", linewidth=1.6, label="True")
ax23.plot(100.0 * x_hat[:, 0], 100.0 * x_hat[:, 1], "b-.", linewidth=1.0, label="Trans")
ax23.plot(100.0 * x_hat_nt[:, 0], 100.0 * x_hat_nt[:, 1], "g-.", linewidth=1.0, label="NoTrans")
ax23.set_title("Movement path")
ax23.set_xlabel("X [cm]")
ax23.set_ylabel("Y [cm]")
ax23.legend(loc="upper right", fontsize=7)
ax23.set_aspect("equal", adjustable="box")

ax24 = fig2.add_subplot(gs[2, 1])
ax24.plot(time, 100.0 * x_true[:, 0], "k", linewidth=1.9)
ax24.plot(time, 100.0 * x_hat[:, 0], "b-.", linewidth=0.9)
ax24.plot(time, 100.0 * x_hat_nt[:, 0], "g-.", linewidth=0.9)
ax24.set_title("X position")

ax25 = fig2.add_subplot(gs[2, 2])
ax25.plot(time, 100.0 * x_true[:, 1], "k", linewidth=1.9)
ax25.plot(time, 100.0 * x_hat[:, 1], "b-.", linewidth=0.9)
ax25.plot(time, 100.0 * x_hat_nt[:, 1], "g-.", linewidth=0.9)
ax25.set_title("Y position")

ax26 = fig2.add_subplot(gs[3, 1])
ax26.plot(time, 100.0 * x_true[:, 2], "k", linewidth=1.9)
ax26.plot(time, 100.0 * x_hat[:, 2], "b-.", linewidth=0.9)
ax26.plot(time, 100.0 * x_hat_nt[:, 2], "g-.", linewidth=0.9)
ax26.set_title("X velocity")
ax26.set_xlabel("time [s]")

ax27 = fig2.add_subplot(gs[3, 2])
ax27.plot(time, 100.0 * x_true[:, 3], "k", linewidth=1.9)
ax27.plot(time, 100.0 * x_hat[:, 3], "b-.", linewidth=0.9)
ax27.plot(time, 100.0 * x_hat_nt[:, 3], "g-.", linewidth=0.9)
ax27.set_title("Y velocity")
ax27.set_xlabel("time [s]")
plt.tight_layout()
plt.show()

rmse = float(np.sqrt(np.mean(err**2)))
rmse_nt = float(np.sqrt(np.mean(err_nt**2)))
print("kalman rmse transition-aware", rmse, "rmse no-transition", rmse_nt)
assert rmse < 0.9


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
